In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
video_path = 'path_to_your_video.mp4'
cap = cv2.VideoCapture(video_path)

ret, first_frame = cap.read()
gray_prev = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)

In [ ]:
# Lucas-Kanade
feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)
lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

p0 = cv2.goodFeaturesToTrack(gray_prev, mask=None, **feature_params)
trajectories = {i: [p0[i].ravel()] for i in range(len(p0))}
point_ids = list(range(len(p0)))
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

for i in range(200):
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    p1, st, err = cv2.calcOpticalFlowPyrLK(gray_prev, gray, p0, None, **lk_params)
    
    if p1 is not None:
        good_new = p1[st == 1]
        good_ids = [point_ids[j] for j in range(len(st)) if st[j]]
        
        for idx, point in zip(good_ids, good_new):
            trajectories[idx].append(point.ravel())
        
        point_ids = good_ids
        p0 = good_new.reshape(-1, 1, 2)
    
    gray_prev = gray.copy()

cap.release()

In [ ]:
# Visualize LK trajectories
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 199)
ret, vis_frame = cap.read()

for point_id, traj in trajectories.items():
    for i in range(len(traj) - 1):
        pt1 = tuple(traj[i].astype(int))
        pt2 = tuple(traj[i + 1].astype(int))
        cv2.line(vis_frame, pt1, pt2, (0, 255, 0), 2)

cap.release()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
# Farneback dense flow
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 50)
ret, frame1 = cap.read()
gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

ret, frame2 = cap.read()
gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

flow = cv2.calcOpticalFlowFarneback(gray1, gray2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])

cap.release()

In [ ]:
# Visualize dense flow
hsv = np.zeros_like(frame1)
hsv[..., 1] = 255
hsv[..., 0] = angle * 180 / np.pi / 2
hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
rgb_flow = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(rgb_flow, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
# Binary motion mask
threshold = 2.0
motion_mask = (magnitude > threshold).astype(np.uint8) * 255

plt.figure(figsize=(12, 8))
plt.imshow(motion_mask, cmap='gray')
plt.axis('off')
plt.show()

## Comparison: Lucas-Kanade vs Farneback

### Observed differences:

**Lucas-Kanade (sparse):**
- Pros: Fast, tracks specific features accurately
- Cons: Loses points over time, needs good texture
- Fails when: Low texture areas, occlusions, objects leave frame

**Farneback (dense):**
- Pros: Complete motion field, detects all moving regions
- Cons: Noisy in low-texture areas, computationally expensive
- Fails when: Uniform regions, shadows, lighting changes

### When to use:

**Use LK when:**
- Need to track specific objects/features
- Speed is critical
- Have distinct keypoints

**Use Farneback when:**
- Need full scene motion analysis
- Detecting all moving objects
- Motion segmentation required

In [ ]:
# Error Analysis: Farneback noise and fragmentation
noise_pixels = np.sum((magnitude > 0) & (magnitude < threshold))
motion_pixels = np.sum(magnitude > threshold)
total_pixels = magnitude.size

print(f"Total pixels: {total_pixels}")
print(f"Motion pixels (>{threshold}): {motion_pixels} ({motion_pixels/total_pixels*100:.1f}%)")
print(f"Noise pixels (0-{threshold}): {noise_pixels} ({noise_pixels/total_pixels*100:.1f}%)")
print(f"Flow magnitude: min={magnitude.min():.2f}, max={magnitude.max():.2f}, mean={magnitude.mean():.2f}")

# Count fragments in binary mask
num_labels, labels = cv2.connectedComponents(motion_mask)
print(f"Number of separate motion regions (fragments): {num_labels - 1}")

In [ ]:
# Error Analysis: LK point loss over time
max_frames = max(len(traj) for traj in trajectories.values())
point_counts = []
for frame_idx in range(max_frames):
    count = sum(1 for traj in trajectories.values() if len(traj) > frame_idx)
    point_counts.append(count)

initial_points = point_counts[0]
final_points = point_counts[-1]
loss_rate = (initial_points - final_points) / initial_points * 100

plt.figure(figsize=(10, 5))
plt.plot(point_counts)
plt.xlabel('Frame')
plt.ylabel('Tracked Points')
plt.title(f'LK Point Loss: {initial_points} → {final_points} ({loss_rate:.1f}% lost)')
plt.grid(True)
plt.show()

print(f"Initial points: {initial_points}")
print(f"Final points: {final_points}")
print(f"Loss rate: {loss_rate:.1f}%")